In [1]:
import torch
import numpy

# Broadcasting
**Goal:** Develop intuition for how PyTorch performs operations on tensors of different shapes without copying data.

## Experiment
Add vectors of different shapes.
  1. create 4 tensors of the following shapes:
```
    tensor1 = (1000, 1)
    tensor2 = (20, 2000)
    tensor3 = (1000,)
    tensor4 = (20, 1182)
```
  2. Try to add all the tensors with each other:
```
     tensor1+tensor2 = succeeds
     tensor1+tensor3 = fails
     tensor3+tensor4 = fails
     tensor1+tensor4 = succeeds
     tensor2+tensor4 = fails
```



In [2]:
# Step 1
tensor1 = torch.rand(1000, 1)
tensor2 = torch.rand(20, 2000)
tensor3 = torch.rand(1000)
tensor4 = torch.rand(20, 1182)

In [3]:
tensor1.shape, tensor2.shape, tensor3.shape, tensor4.shape

(torch.Size([1000, 1]),
 torch.Size([20, 2000]),
 torch.Size([1000]),
 torch.Size([20, 1182]))

In [4]:
# Step2
tensor1+tensor2

RuntimeError: The size of tensor a (1000) must match the size of tensor b (20) at non-singleton dimension 0

In [5]:
tensor1+tensor3

tensor([[1.1172, 0.8218, 0.7892,  ..., 1.0405, 0.8600, 0.4898],
        [1.2685, 0.9731, 0.9405,  ..., 1.1918, 1.0113, 0.6410],
        [1.1597, 0.8643, 0.8317,  ..., 1.0830, 0.9025, 0.5323],
        ...,
        [1.1218, 0.8264, 0.7938,  ..., 1.0451, 0.8646, 0.4943],
        [1.0964, 0.8010, 0.7684,  ..., 1.0197, 0.8392, 0.4689],
        [1.0287, 0.7333, 0.7007,  ..., 0.9520, 0.7715, 0.4013]])

In [6]:
(tensor1+tensor3).shape

torch.Size([1000, 1000])

In [7]:
tensor3+tensor4

RuntimeError: The size of tensor a (1000) must match the size of tensor b (1182) at non-singleton dimension 1

In [8]:
tensor1+tensor4

RuntimeError: The size of tensor a (1000) must match the size of tensor b (20) at non-singleton dimension 0

In [9]:
tensor2+tensor4

RuntimeError: The size of tensor a (2000) must match the size of tensor b (1182) at non-singleton dimension 1

## Rule:
1. When 2 tensors have different number of dimensions, add 1s to the left of the smaller tensor
2. Then, starting from right, compare the size of each dimension between each tensor:
  - Both have equal size -> valid
  - One has size 1 and the other has size n -> compatible, but PyTorch conceptually expands the smaller tensor to size n without allocating additional memory.
  - Both are different and neither is 1 -> invalid

## Observation



1. tensor1+tensor2 = failed :

    tensor1.shape = (1000, 1)
    tensor2.shape = (20, 2000)

    Comparison from right:
      - dim1: 1 vs 2000 -> compatible
      - dim0 = 1000 vs 20 -> incompatible
  
2. tensor1+tensor3 = succeeded :

    tensor1.shape = (1000, 1)
    tensor3.shape = (1000,)

    Adding 1s to the start of tensor3's dimensions to make the number of dims equal

    tensor1.shape = (1000, 1)
    tensor3.shape = (1, 1000)

    Comparison from right:
      - dim1: 1 vs 1000 -> compatible
      - dim0 = 1000 vs 1 -> compatible

3. tensor3+tensor4 = fails :

    tensor3.shape = (1000,)
    tensor4.shape = (20, 1182)

    Adding 1s to the start of tensor3's dimensions to make the number of dims equal

    tensor3.shape = (1, 1000)
    tensor4.shape = (20, 1182)

    Comparison from right:
      - dim1: 1000 vs 1182 -> incompatible

4. tensor1+tensor4 = fails :

    tensor1.shape = (1000, 1)
    tensor4.shape = (20, 1182)

    Comparison from right:
      - dim1: 1 vs 1182 -> compatible
      - dim0 = 1000 vs 20 -> incompatible

5. tensor2+tensor4 = fails :

    tensor2.shape = (20, 2000)
    tensor4.shape = (20, 1182)

    Comparison from right:
      - dim1: 2000 vs 1182 -> incompatible


# Key Takeaways
- Broadcasting compares shapes from the rightmost dimension.
- Missing leading dimensions are treated as size 1.
- A dimension is compatible if the sizes are equal or one of them is 1.
- Broadcasting behaves as if size-1 dimensions were expanded, avoiding unnecessary data duplication.
- Broadcasting is used extensively in deep learning, especially for bias addition and element-wise operations.